# Solution D3 · Anscombe: four datasets, one line, three wrong models

**Type**: Interpret

## Objective

Verify that the four datasets of Anscombe's quartet have the same summary statistics, and then use leverage and
Cook's distance to understand which point moves the line in each one.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression

anscombe = ds.anscombe()
sets = {name: g[["x", "y"]].reset_index(drop=True) for name, g in anscombe.groupby("dataset")}
sets["I"].head(3)

## Commented solution

In [ ]:
rows = {}
for name, d in sets.items():
    m = LinearRegression().fit(d[["x"]], d["y"])
    residuals = d["y"] - m.predict(d[["x"]])
    rows[name] = {"slope": m.coef_[0], "intercept": m.intercept_, "r2": m.score(d[["x"]], d["y"]),
                  "resid_std": residuals.std(ddof=2)}
summary = pd.DataFrame(rows).T
summary.round(3)

The four rows are practically identical: anyone who limits themselves to these statistics would conclude that the four
datasets are the same problem. Only Cook's distance and the plots show the difference.

In [ ]:
def leverages(x):
    A = np.column_stack([np.ones(len(x)), x])
    return np.diag(A @ np.linalg.inv(A.T @ A) @ A.T)


def cook(x, y):
    A = np.column_stack([np.ones(len(x)), x])
    n, p = A.shape
    e = y - A @ np.linalg.lstsq(A, y, rcond=None)[0]
    s2 = e @ e / (n - p)
    h = leverages(x)
    with np.errstate(all="ignore"):
        d = e ** 2 / (p * s2) * h / (1 - h) ** 2
    return np.where(h > 1 - 1e-9, np.inf, d)

In [ ]:
influential_point = {}
for name, d in sets.items():
    x, y = d["x"].to_numpy(float), d["y"].to_numpy(float)
    influential_point[name] = int(np.argmax(cook(x, y)))
print(influential_point)
print("maximum leverage in IV:", leverages(sets["IV"].x.to_numpy(float)).max())

In [ ]:
d3 = sets["III"].drop(index=influential_point["III"])
m3 = LinearRegression().fit(d3[["x"]], d3["y"])
slope_III_without = m3.coef_[0]
r2_III_without = m3.score(d3[["x"]], d3["y"])

d4 = sets["IV"].drop(index=influential_point["IV"])
iv_x_constant = d4["x"].nunique() == 1
print(f"III without the outlier: slope {slope_III_without:.3f}, R2 {r2_III_without:.4f}")
print("IV without the isolated point: are all x equal?", iv_x_constant)

In [ ]:
regression_suitable = {"I": True, "II": False, "III": False, "IV": False}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6.5), sharex=True, sharey=True)
for a, (name, d) in zip(axes.ravel(), sets.items()):
    m = LinearRegression().fit(d[["x"]], d["y"])
    g = np.linspace(3, 20, 50)
    a.scatter(d.x, d.y, s=25)
    a.plot(g, m.predict(pd.DataFrame({"x": g})), color="#1F6E5C")
    i = influential_point[name]
    a.scatter(d.x[i], d.y[i], s=90, facecolors="none", edgecolors="#9C4530", lw=2)
    a.set_title(f"Dataset {name}")
fig.tight_layout()

## Check

In [ ]:
with verify():
    check(list(summary.columns) == ["slope", "intercept", "r2", "resid_std"],
          "the summary has the required columns", "columns: slope, intercept, r2, resid_std")
    check(np.allclose(summary.slope, 0.5, atol=0.01) and np.allclose(summary.intercept, 3.0, atol=0.01),
          "slope and intercept are practically identical in the four datasets (0.5 and 3.0)",
          "the four lines should have slope 0.5 and intercept 3.0")
    check(np.allclose(summary.r2, 0.667, atol=0.01), "the R2 is identical (0.667) in the four datasets",
          "the R2 should be about 0.667 in all four")
    check(abs(leverages(sets["IV"].x.to_numpy(float)).max() - 1) < 1e-9, "in dataset IV one point has leverage 1",
          "the point with x = 19 is the only one with that value: its leverage is 1")
    check(np.isinf(cook(sets["IV"].x.to_numpy(float), sets["IV"].y.to_numpy(float))).sum() == 1,
          "the Cook's distance of that point is infinite", "assign np.inf to the points with leverage > 1 - 1e-9")
    check(influential_point["III"] == 2 and influential_point["IV"] == 7,
          "influential points: index 2 in III (the vertical outlier), index 7 in IV (the isolated point)",
          "in III the outlier has x = 13, y = 12.74 (index 2); in IV the point with x = 19 has index 7")
    check(abs(slope_III_without - 0.345) < 0.01 and r2_III_without > 0.99,
          "without the outlier dataset III is almost perfectly linear (slope 0.345, R2 ~ 1)",
          "remove the point with index 2 from dataset III and refit")
    check(bool(iv_x_constant), "in dataset IV, without that point, x is constant and the slope is not identifiable",
          "after removing the point x = 19 all the x equal 8")
    check(regression_suitable == {"I": True, "II": False, "III": False, "IV": False}, "correct judgment on the four datasets",
          "only dataset I is a normal linear cloud: II is curved, III has an outlier, IV depends on one point")
done()

## What you learned

- The summary statistics (slope, intercept, R²) do not identify the model: four very different structures give
  the same numbers. Looking at the data is not optional.
- **II** is a parabola: the line is the wrong model (violation of linearity, visible in the residuals). **III** is a
  perfect line with a single outlier, which by itself explains the wrong slope: remove it and R² goes up to 1.
- **IV** is the limit case: a single point with an x different from the others determines the whole slope. It has leverage exactly 1: the
  line passes *necessarily* through the point (residual 0), and Cook's distance is 0/0. Without that point x is constant and
  the slope is not even defined: the data contain no information about the x-y relationship.

## Common pitfall

Saying "I remove the outlier" without understanding whether it is a measurement error or a piece of reality. In III the point is probably an
error (it must be corrected or investigated); in IV the isolated point **is the only information** we have about the relationship, and
removing it means no longer having a model.

## Interview question

<details><summary>In your model a single point has a Cook's distance 40 times the threshold. Do you remove it?</summary>

First I understand why: I look at its leverage and its residual, and check whether it is a recording error. If it is an error I
correct it or remove it documenting the choice. If it is a real but extreme observation (for example a customer much
bigger than the others), I would in effect be removing the part of the range that I want to predict: I prefer a robust model (Huber),
a transformation that reduces the leverage, or I state that the model does not cover that area. I always report the results
with and without the point.

</details>